In [1]:
import json
from pathlib import Path
import torch
from scgpt.model import TransformerModel
from scgpt.tokenizer import GeneVocab
from scgpt.data_collator import DataCollator
from torch.utils.data import DataLoader, SequentialSampler
import anndata as an
import numpy as np
import scanpy as sc
import os
from tqdm import tqdm
import pandas as pd
import gc

/software/envs/micromamba/envs/nw-scGPTv2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Definizione dei percorsi
model_dir = Path("/projects/shared/intronic_bam/scGPT_supplements/save/scGPT_human")
model_config_file = model_dir / "args.json"
model_file = model_dir / "best_model.pt"
vocab_file = model_dir / "vocab.json"
save_dir = Path("/projects/shared/intronic_bam/datasets/embeddings/scgpt_layers/be1")

batch_size = 8

In [3]:
# 2. Caricamento del vocabolario dei geni
vocab = GeneVocab.from_file(vocab_file)
special_tokens = ["<pad>", "<cls>", "<eoc>", "<mask>", "<unk>"]
for s in special_tokens:
    if s not in vocab:
        vocab.append_token(s)

# 3. Caricamento della configurazione dell'architettura
with open(model_config_file, "r") as f:
    model_configs = json.load(f)

# 4. Inizializzazione del modello Transformer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerModel(
    ntoken=len(vocab),
    d_model=model_configs["embsize"],
    nhead=model_configs["nheads"],
    d_hid=model_configs["d_hid"],
    nlayers=model_configs["nlayers"],
    nlayers_cls=model_configs.get("n_layers_cls", 3),
    vocab=vocab,
    pad_token="<pad>",  # Passa la stringa "<pad>", non l'indice vocab["<pad>"]
    pad_value=model_configs.get("pad_value", 0),
    do_mvc=model_configs.get("GEPC", False),
)


In [4]:
# 5. Caricamento dei pesi (state dict)
model_dict = model.state_dict()
pretrained_dict = torch.load(model_file, map_location=device)
pretrained_dict = {
    k: v for k, v in pretrained_dict.items() 
    if k in model_dict and v.shape == model_dict[k].shape
}
model_dict.update(pretrained_dict)
model.load_state_dict(model_dict)

model.to(device)
model.eval()

print("Modello caricato con successo su:", device)


Modello caricato con successo su: cuda


In [5]:
class Dataset(torch.utils.data.Dataset):
        def __init__(self, count_matrix, gene_ids, batch_ids=None):
            self.count_matrix = count_matrix
            self.gene_ids = gene_ids
            self.batch_ids = batch_ids

        def __len__(self):
            return len(self.count_matrix)

        def __getitem__(self, idx):
            row = self.count_matrix[idx]
            nonzero_idx = np.nonzero(row)[0]
            values = row[nonzero_idx]
            genes = self.gene_ids[nonzero_idx]
            # append <cls> token at the beginning
            genes = np.insert(genes, 0, vocab["<cls>"])
            values = np.insert(values, 0, model_configs["pad_value"])
            genes = torch.from_numpy(genes).long()
            values = torch.from_numpy(values).float()
            output = {
                "id": idx,
                "genes": genes,
                "expressions": values,
            }
            if self.batch_ids is not None:
                output["batch_labels"] = self.batch_ids[idx]
            return output

In [6]:
adata = an.read_h5ad("/projects/shared/intronic_bam/datasets/anndata/be1.h5ad")

In [7]:
adata.X = adata.layers["counts"].copy()

In [8]:
#adata = adata[:100, :].copy()

In [8]:
gene_col = "gene_names"
adata.var["id_in_vocab"] = [
        vocab[gene] if gene in vocab else -1 for gene in adata.var[gene_col]
    ]
gene_ids_in_vocab = np.array(adata.var["id_in_vocab"])

adata = adata[:, adata.var["id_in_vocab"] >= 0]

with open(model_config_file, "r") as f:
        model_configs = json.load(f)

# Binning will be applied after tokenization. A possible way to do is to use the unified way of binning in the data collator.

vocab.set_default_index(vocab["<pad>"])
genes = adata.var[gene_col].tolist()
gene_ids = np.array(vocab(genes), dtype=int)

In [9]:
count_matrix = adata.X
count_matrix = (
    count_matrix if isinstance(count_matrix, np.ndarray) else count_matrix.toarray()
)

# gene vocabulary ids
if gene_ids is None:
    gene_ids = np.array(adata.var["id_in_vocab"])
    assert np.all(gene_ids >= 0)

use_batch_labels = False # cosi fanno loro
if use_batch_labels:
    batch_ids = np.array(adata.obs["batch_id"].tolist())

dataset = Dataset(
            count_matrix, gene_ids, batch_ids if use_batch_labels else None
        )

In [10]:
collator = DataCollator(
            do_padding=True,
            pad_token_id=vocab[model_configs["pad_token"]],
            pad_value=model_configs["pad_value"],
            do_mlm=False,
            do_binning=True,
            max_length=1200,
            sampling=True,
            keep_first_n_tokens=1,
    )
data_loader = DataLoader(
            dataset,
            batch_size=batch_size,
            sampler=SequentialSampler(dataset),
            collate_fn=collator,
            drop_last=False,
            num_workers=1,
            pin_memory=True,
)


In [11]:
batch_prova = next(iter(data_loader))

In [ ]:
emb_layers = {}
hooks = []

def get_layer_embed(name):
    def hook(module, input, output):
        # Se l'output è una tupla/lista (es. (tensor, attn_weights)), estraiamo il primo elemento
        if isinstance(output, (tuple, list)):
            out_tensor = output[0]
        else:
            out_tensor = output
            
        # Salva solo se è un vero tensore PyTorch
        if isinstance(out_tensor, torch.Tensor):
            emb_layers[name] = out_tensor.detach().cpu()
    return hook

# 1. Registra l'hook SOLO sui blocchi principali dell'encoder
for name, module in model.named_modules():
    if "transformer_encoder.layers." in name and name.split(".")[-1].isdigit():
        hooks.append(module.register_forward_hook(get_layer_embed(name)))
print(f"Hook registrati su {len(hooks)} layer principali.")

In [ ]:
# 2. Esegui il forward pass per un singolo pezzo di dati
input_gene_ids = batch_prova["gene"].to(device)
src_key_padding_mask = input_gene_ids.eq(
    vocab[model_configs["pad_token"]]
)
with torch.no_grad():
    output = model(input_gene_ids, batch_prova["expr"].to(device), src_key_padding_mask)

In [ ]:
# 3. Verifica i layer catturati
for name, tensor in emb_layers.items():
    print(f"Layer: {name}")

# 4. Rimuovi gli hook per liberare la memoria
for h in hooks:
    h.remove()


In [16]:
emb_layers['transformer_encoder.layers.11']

nested_tensor([
  tensor([[ 0.3684,  0.3175,  0.1220,  ...,  0.3516, -0.0885,  0.3568],
          [ 0.4266,  0.3121,  0.1308,  ...,  0.3536, -0.0979,  0.3355],
          [ 0.4230,  0.3381,  0.1094,  ...,  0.3796, -0.1591,  0.2922],
          ...,
          [ 0.3991,  0.3097,  0.1190,  ...,  0.3716, -0.1028,  0.3581],
          [ 0.4125,  0.3227,  0.1153,  ...,  0.3717, -0.1315,  0.3115],
          [ 0.4166,  0.3332,  0.1420,  ...,  0.3613, -0.1100,  0.3390]]),
  tensor([[ 0.3698,  0.3030,  0.1398,  ...,  0.3236, -0.1295,  0.3565],
          [ 0.3992,  0.3111,  0.1792,  ...,  0.3373, -0.1443,  0.3470],
          [ 0.4021,  0.3010,  0.2009,  ...,  0.3543, -0.1594,  0.2889],
          ...,
          [ 0.4104,  0.3048,  0.1819,  ...,  0.3308, -0.1673,  0.3122],
          [ 0.4029,  0.3125,  0.1683,  ...,  0.3348, -0.1802,  0.2933],
          [ 0.3962,  0.3235,  0.1609,  ...,  0.3361, -0.1652,  0.3514]]),
  tensor([[ 0.3822,  0.3146,  0.1197,  ...,  0.3007, -0.0838,  0.3455],
          [ 0.

In [17]:
emb_layers['transformer_encoder.layers.11'][0].shape

torch.Size([1200, 512])

In [18]:
embeddings = model._encode(
                    input_gene_ids,
                    batch_prova["expr"].to(device),
                    src_key_padding_mask=src_key_padding_mask,
                )

In [19]:
embeddings.shape

torch.Size([8, 1200, 512])

In [11]:
def get_all_layer_embeddings(model, data_loader, vocab, model_configs,
                              use_batch_labels=False, normalize=True):
    device = next(model.parameters()).device

    # --- Stesso hook che già usi ---
    emb_layers = {}
    hooks = []

    def get_layer_embed(name):
        def hook(module, inp, out):
            out = out[0] if isinstance(out, (tuple, list)) else out
            if isinstance(out, torch.Tensor):
                # estrai subito solo il CLS token → [bs, 512] invece di [bs, 1200, 512]
                emb_layers[name] = out[:, 0, :].detach().cpu()
        return hook


    for name, module in model.named_modules():
        if "transformer_encoder.layers." in name and name.split(".")[-1].isdigit():
            hooks.append(module.register_forward_hook(get_layer_embed(name)))

    # --- Itera il dataloader ---
    collected = {}   # {layer_name: [array_batch1, array_batch2, ...]}

    try:
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
            for data_dict in tqdm(data_loader, desc="Embedding cells"):
                emb_layers.clear()

                input_gene_ids = data_dict["gene"].to(device)
                src_key_padding_mask = input_gene_ids.eq(vocab[model_configs["pad_token"]])

                model._encode(
                    input_gene_ids,
                    data_dict["expr"].to(device),
                    src_key_padding_mask=src_key_padding_mask,
                    batch_labels=data_dict["batch_labels"].to(device) if use_batch_labels else None,
                )
                gc.collect()
                torch.cuda.empty_cache()
                for name, tensor in emb_layers.items():
                    collected.setdefault(name, []).append(tensor.numpy())  # già [bs, 512]

    finally:
        for h in hooks:
            h.remove()

    # --- Concatena e ordina per layer index ---
    layer_names = sorted(collected, key=lambda n: int(n.split(".")[-1]))
    # [n_cells, n_layers, 512]
    all_embs = np.stack(
        [np.concatenate(collected[n], axis=0) for n in layer_names], axis=1
    )

    if normalize:
        norms = np.linalg.norm(all_embs, axis=2, keepdims=True)
        all_embs /= np.maximum(norms, 1e-8)

    return all_embs, layer_names

In [12]:
# tutti i layer
embs, layer_names = get_all_layer_embeddings(model, data_loader, vocab, model_configs)
embs.shape  # (n_cells, 12, 512)


/tmp/ipykernel_2959597/3285197882.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
Embedding cells: 100%|██████████| 3641/3641 [15:34<00:00,  3.90it/s]


(29128, 12, 512)

In [13]:
for i, lname in enumerate(layer_names):
    layer_idx = lname.split(".")[-1]
    df = pd.DataFrame(embs[:, i, :], index=adata.obs.index)
    df.index.name = "Barcode"
    df.to_csv(save_dir / f"be1_scgpt_layer_{layer_idx}.csv")